In [1]:
import pandas as pd, numpy as np, json
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report

D     = r"C:\Users\Bhasura\Desktop\4th year Research\dataset research\com 2\archive2\Edge-IIoTset dataset\Selected dataset for ML and DL"      
TRAIN = D + r"\EdgeIIoT_train_80.csv"
TEST  = D + r"\EdgeIIoT_test_20.csv"
FULL  = D + r"\ML-EdgeIIoT-dataset.csv"

tr = pd.read_csv(TRAIN, low_memory=False)
te = pd.read_csv(TEST,  low_memory=False)
print(tr.shape, te.shape) 

(126240, 62) (31560, 62)


In [2]:
full = pd.read_csv(FULL, low_memory=False)
print("full:", full.shape, "| train+test:", len(tr) + len(te))

cols = list(tr.columns)
print("columns missing in full:", set(cols) - set(full.columns))

fs = full[cols].astype(str)
fs['Attack_type'] = full['Attack_type'].astype(str).str.strip()
fs = fs.drop_duplicates()
amb = fs.duplicated(subset=cols, keep=False)      # එකම features, වෙනස් types
print("ambiguous feature-rows:", amb.sum())
fs = fs[~amb]

for name, d in [('train', tr), ('test', te)]:
    m = d[cols].astype(str).merge(fs, on=cols, how='left')
    assert len(m) == len(d)
    d['Attack_type'] = m['Attack_type'].values
    print(name, "unlabeled:", d['Attack_type'].isna().sum())

print(tr['Attack_type'].value_counts())
print(te['Attack_type'].value_counts())

full: (157800, 63) | train+test: 157800
columns missing in full: set()
ambiguous feature-rows: 0
train unlabeled: 19416
test unlabeled: 4669
Attack_type
Normal                   11553
DDoS_UDP                 11534
Ransomware                8658
DDoS_ICMP                 8356
SQL_injection             8264
DDoS_TCP                  8261
Uploading                 8173
Backdoor                  8121
XSS                       8076
Vulnerability_scanner     8043
Port_Scanning             8015
Password                  7986
MITM                       978
Fingerprinting             806
Name: count, dtype: int64
Attack_type
Normal                   2971
DDoS_UDP                 2964
Ransomware               2267
Uploading                2096
Backdoor                 2074
Port_Scanning            2056
SQL_injection            2047
Vulnerability_scanner    2032
Password                 2003
DDoS_ICMP                1988
DDoS_TCP                 1986
XSS                      1976
MITM           

In [3]:
cols = [c for c in tr.columns if c != 'Attack_type']

# dtype වෙනස් columns (diagnostic)
print("dtype differs:", [c for c in cols if tr[c].dtype != full[c].dtype])

def norm(df):
    out = {}
    for c in cols:
        s = df[c]
        n = pd.to_numeric(s, errors='coerce')
        txt = s.astype(str).str.strip()
        out[c] = np.where(n.notna(), n.astype(float).astype(str), txt)   # 0 / 0.0 / "0.0" -> '0.0'
    return pd.DataFrame(out, index=df.index)

fs = norm(full)
fs['Attack_type'] = full['Attack_type'].astype(str).str.strip().values
fs = fs.drop_duplicates()
amb = fs.duplicated(subset=cols, keep=False)
print("ambiguous feature-rows:", amb.sum())
fs = fs[~amb]

for name, d in [('train', tr), ('test', te)]:
    m = norm(d).merge(fs, on=cols, how='left')
    assert len(m) == len(d)
    d['Attack_type'] = m['Attack_type'].values
    print(name, "unlabeled:", d['Attack_type'].isna().sum())

print(tr['Attack_type'].value_counts())
print(te['Attack_type'].value_counts())

dtype differs: []
ambiguous feature-rows: 0
train unlabeled: 0
test unlabeled: 0
Attack_type
Normal                   19441
DDoS_UDP                 11534
DDoS_ICMP                11357
Ransomware                8658
DDoS_HTTP                 8526
SQL_injection             8264
DDoS_TCP                  8261
Uploading                 8173
Backdoor                  8121
XSS                       8076
Vulnerability_scanner     8044
Port_Scanning             8015
Password                  7986
MITM                       978
Fingerprinting             806
Name: count, dtype: int64
Attack_type
Normal                   4860
DDoS_UDP                 2964
DDoS_ICMP                2733
Ransomware               2267
Uploading                2096
Backdoor                 2074
Port_Scanning            2056
SQL_injection            2047
DDoS_HTTP                2035
Vulnerability_scanner    2032
Password                 2003
DDoS_TCP                 1986
XSS                      1976
MITM          

features: 59 | test: (31560, 59)


ValueError: feature_names must be string, and may not contain [, ] or <

In [5]:
drop_cols = ['frame.time','ip.src_host','ip.dst_host','arp.dst.proto_ipv4',
             'arp.src.proto_ipv4','http.file_data','http.request.uri.query',
             'http.referer','http.request.full_uri','tcp.payload','mqtt.msg',
             'Attack_label','Attack_type']
X_tr = tr.drop(columns=drop_cols, errors='ignore').copy()
X_te = te.drop(columns=drop_cols, errors='ignore').copy()

new_tr, new_te = {}, {}
for c in list(X_tr.columns):
    if pd.api.types.is_numeric_dtype(X_tr[c]):
        X_tr[c] = X_tr[c].fillna(0); X_te[c] = X_te[c].fillna(0); continue
    a = pd.to_numeric(X_tr[c], errors='coerce')
    if a.notna().mean() > 0.99:                       # numeric වගේ text columns (tcp.srcport...)
        X_tr[c] = a.fillna(0); X_te[c] = pd.to_numeric(X_te[c], errors='coerce').fillna(0)
    else:
        s_tr = X_tr[c].astype(str).str.strip(); s_te = X_te[c].astype(str).str.strip()
        cats = sorted(s_tr.unique())
        if len(cats) > 50:                            # high-cardinality -> label code (unseen = -1)
            mp = {v: i for i, v in enumerate(cats)}
            X_tr[c] = s_tr.map(mp); X_te[c] = s_te.map(mp).fillna(-1)
        else:                                         # low-cardinality -> one-hot (train categories)
            for k in cats:
                new_tr[f'{c}_{k}'] = (s_tr == k).astype(int)
                new_te[f'{c}_{k}'] = (s_te == k).astype(int)
            X_tr = X_tr.drop(columns=c); X_te = X_te.drop(columns=c)

X_tr = pd.concat([X_tr, pd.DataFrame(new_tr, index=X_tr.index)], axis=1)
X_te = pd.concat([X_te, pd.DataFrame(new_te, index=X_te.index)], axis=1)
X_te = X_te.reindex(columns=X_tr.columns, fill_value=0)   # test columns = train columns, same order
print("features:", X_tr.shape[1], "| test:", X_te.shape)

# train: labeled rows විතරක්
ok = tr['Attack_type'].notna().values
le = LabelEncoder().fit(tr.loc[ok, 'Attack_type'])
y_tr = le.transform(tr.loc[ok, 'Attack_type'])

model = XGBClassifier(n_estimators=100, objective='multi:softprob',
                      tree_method='hist', n_jobs=-1, random_state=42)
model.fit(X_tr[ok], y_tr)

proba = model.predict_proba(X_te)
pred  = proba.argmax(axis=1)
out = pd.DataFrame({'row': np.arange(len(te)),            # 0-based, test_20 ordering එකම
                    'predicted_type': le.inverse_transform(pred),
                    'confidence': proba.max(axis=1).round(6)})
out.to_csv('xgb_predictions_test20.csv', index=False)
print(out.shape); print(out.head())

# metrics (test labels තියෙන rows වලට විතරයි)
m = te['Attack_type'].notna().values
yt, yp = te.loc[m, 'Attack_type'].values, out.loc[m, 'predicted_type'].values
print("accuracy :", accuracy_score(yt, yp))
print("macro-F1 :", f1_score(yt, yp, average='macro', labels=sorted(set(yt))))
print(classification_report(yt, yp, digits=4))
json.dump(list(X_tr.columns), open('feature_names_15class.json', 'w'))
print("classes:", list(le.classes_))

features: 59 | test: (31560, 59)


ValueError: feature_names must be string, and may not contain [, ] or <

In [6]:
import re

# 1) ඒ අකුරු තියෙන names clean කරන්න (train columns වලින් mapping, test එකටත් එකම mapping)
X_te = X_te.reindex(columns=X_tr.columns, fill_value=0)
seen, newcols = {}, []
for c in X_tr.columns:
    n = re.sub(r'[\[\]<>]', '_', str(c))
    if n in seen:
        seen[n] += 1; n = f"{n}__{seen[n]}"
    else:
        seen[n] = 0
    newcols.append(n)
X_tr.columns = newcols
X_te.columns = newcols
print("features:", X_tr.shape[1], "| unique names:", X_tr.columns.is_unique)

# 2) train (train_80 එකෙන් විතරක්)
model = XGBClassifier(n_estimators=100, objective='multi:softprob',
                      tree_method='hist', n_jobs=-1, random_state=42)
model.fit(X_tr[ok], y_tr)

# 3) test_20 predictions (31,560 rows, original order)
proba = model.predict_proba(X_te)
pred  = proba.argmax(axis=1)
out = pd.DataFrame({'row': np.arange(len(te)),
                    'predicted_type': le.inverse_transform(pred),
                    'confidence': proba.max(axis=1).round(6)})
out.to_csv('xgb_predictions_test20.csv', index=False)
print(out.shape); print(out.head())

# 4) metrics
m = te['Attack_type'].notna().values
yt, yp = te.loc[m, 'Attack_type'].values, out.loc[m, 'predicted_type'].values
print("accuracy :", accuracy_score(yt, yp))
print("macro-F1 :", f1_score(yt, yp, average='macro', labels=sorted(set(yt))))
print(classification_report(yt, yp, digits=4))
json.dump(list(X_tr.columns), open('feature_names_15class.json', 'w'))
print("classes:", list(le.classes_))

features: 59 | unique names: True
(31560, 3)
   row predicted_type  confidence
0    0       DDoS_UDP    0.999983
1    1         Normal    0.999997
2    2       DDoS_UDP    0.999983
3    3            XSS    0.995774
4    4       DDoS_UDP    0.999980
accuracy : 0.6636248415716096
macro-F1 : 0.6391736612448881
                       precision    recall  f1-score   support

             Backdoor     1.0000    0.0535    0.1016      2074
            DDoS_HTTP     0.5173    0.5356    0.5263      2035
            DDoS_ICMP     1.0000    0.9996    0.9998      2733
             DDoS_TCP     0.8650    1.0000    0.9276      1986
             DDoS_UDP     1.0000    1.0000    1.0000      2964
       Fingerprinting     0.9938    0.8256    0.9020       195
                 MITM     1.0000    1.0000    1.0000       236
               Normal     0.4672    0.9963    0.6361      4860
             Password     0.2766    0.1313    0.1781      2003
        Port_Scanning     0.9837    1.0000    0.9918      20

In [7]:
txt_cols = ['http.file_data','http.request.uri.query','http.referer',
            'http.request.full_uri','tcp.payload','mqtt.msg']

def add_txt(X, src, maps):
    X = X.copy()
    for c in txt_cols:
        s = src[c].astype(str).str.strip()
        X[f'{c}_len']  = s.str.len().values
        X[f'{c}_code'] = s.map(maps[c]).fillna(-1).values
    return X

maps = {}
for c in txt_cols:
    s = tr[c].astype(str).str.strip()
    vc = s.value_counts()
    maps[c] = {v: i for i, v in enumerate(vc[vc >= 5].index)}   # train එකෙන් විතරයි

X_tr2 = add_txt(X_tr, tr, maps)
X_te2 = add_txt(X_te, te, maps)
print("features:", X_tr2.shape[1])

model2 = XGBClassifier(n_estimators=100, objective='multi:softprob',
                       tree_method='hist', n_jobs=-1, random_state=42)
model2.fit(X_tr2[ok], y_tr)

proba2 = model2.predict_proba(X_te2)
out2 = pd.DataFrame({'row': np.arange(len(te)),
                     'predicted_type': le.inverse_transform(proba2.argmax(axis=1)),
                     'confidence': proba2.max(axis=1).round(6)})
out2.to_csv('xgb_predictions_test20_v2.csv', index=False)

m = te['Attack_type'].notna().values
yt, yp = te.loc[m, 'Attack_type'].values, out2.loc[m, 'predicted_type'].values
print("accuracy :", accuracy_score(yt, yp))
print("macro-F1 :", f1_score(yt, yp, average='macro', labels=sorted(set(yt))))
print(classification_report(yt, yp, digits=4))
json.dump(list(X_tr2.columns), open('feature_names_15class_v2.json', 'w'))

features: 71
accuracy : 0.9064955640050697
macro-F1 : 0.8896306317494371
                       precision    recall  f1-score   support

             Backdoor     0.6947    0.9730    0.8106      2074
            DDoS_HTTP     0.5093    0.8486    0.6366      2035
            DDoS_ICMP     1.0000    0.9996    0.9998      2733
             DDoS_TCP     1.0000    1.0000    1.0000      1986
             DDoS_UDP     1.0000    1.0000    1.0000      2964
       Fingerprinting     0.9938    0.8256    0.9020       195
                 MITM     1.0000    1.0000    1.0000       236
               Normal     0.9998    1.0000    0.9999      4860
             Password     0.5291    0.1727    0.2604      2003
        Port_Scanning     0.9837    1.0000    0.9918      2056
           Ransomware     0.9610    0.6087    0.7453      2267
        SQL_injection     1.0000    0.9995    0.9998      2047
            Uploading     1.0000    1.0000    1.0000      2096
Vulnerability_scanner     1.0000    0.9966  